# 02 — Калибровка и оценка до PPO

Ноутбук фиксирует seed-разбиение, проверяет контракт Gymnasium, калибрует синтетический рынок и сравнивает baseline-стратегии на парных траекториях. `QUICK_RUN=True` предназначен для проверки кода; итоговые выводы делаются только при полном запуске.

In [ ]:
from pathlib import Path
import sys

try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path('/content/drive/MyDrive/gas_portfolio_rl')
except ImportError:
    IN_COLAB = False
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = cwd if (cwd / 'pyproject.toml').exists() else cwd / 'gas_portfolio_rl'

sys.path.insert(0, str(PROJECT_ROOT))
from notebook_bootstrap import bootstrap_project
PROJECT_ROOT = bootstrap_project(PROJECT_ROOT, install_dependencies=IN_COLAB)

import gas_portfolio_rl
print('Runtime:', 'Google Colab' if IN_COLAB else 'local')
print('Project root:', PROJECT_ROOT)
print('Package imported from:', gas_portfolio_rl.__file__)

In [ ]:
from dataclasses import replace
import json
from gas_portfolio_rl import AppConfig
from gas_portfolio_rl.calibration import calibrate_market, save_calibration
from gas_portfolio_rl.evaluation import (
    build_reward_gate, evaluate_strategies, make_default_strategies,
    make_ppo_feasible_grid_strategies, make_scenarios, save_reward_gate,
)
from gas_portfolio_rl.evaluation.preflight import run_preflight_checks
from gas_portfolio_rl.evaluation.reporting import save_evaluation
from gas_portfolio_rl.evaluation.splits import SeedManifest
from IPython.display import Image, display

QUICK_RUN = True  # После smoke-run переключите на False для итоговой оценки.
MODEL_PROFILE = 'ppo_v5_full.yaml'  # Reward gate проверяется в том же action-space, что и PPO.
config = AppConfig.from_yaml(
    PROJECT_ROOT / 'configs/market/default.yaml',
    PROJECT_ROOT / 'configs/environment/default.yaml',
    PROJECT_ROOT / 'configs/reward/default.yaml',
    PROJECT_ROOT / 'configs/baselines/default.yaml',
    PROJECT_ROOT / 'configs/calibration/default.yaml',
    PROJECT_ROOT / 'configs/evaluation/default.yaml',
    PROJECT_ROOT / 'configs/models' / MODEL_PROFILE,
)
if QUICK_RUN:
    config = replace(
        config,
        calibration=replace(config.calibration, paths_per_candidate=2, candidate_count=2, path_steps=60),
        evaluation=replace(config.evaluation, seeds=3, stress_seeds=2, reward_grid_seeds=2, bootstrap_samples=30),
    )
OUTPUT_DIR = PROJECT_ROOT / 'results/pre_ppo'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
manifest = SeedManifest().as_dict()
preflight = run_preflight_checks(config)
(OUTPUT_DIR / 'seed_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'preflight.json').write_text(json.dumps(preflight, indent=2), encoding='utf-8')
run_metadata = {
    'quick_run': QUICK_RUN,
    'calibration_paths_per_candidate': config.calibration.paths_per_candidate,
    'calibration_candidate_count': config.calibration.candidate_count,
    'validation_seeds': config.evaluation.seeds,
    'stress_seeds': config.evaluation.stress_seeds,
    'final_evaluation': not QUICK_RUN,
    'ppo_experiment_tag': config.ppo.experiment_tag,
    'model_profile': MODEL_PROFILE,
}
(OUTPUT_DIR / 'run_metadata.json').write_text(json.dumps(run_metadata, indent=2), encoding='utf-8')
print(manifest)
print(preflight)

In [ ]:
calibration = calibrate_market(config.market, config.calibration)
config = replace(config, market=calibration.market_config)
save_calibration(calibration, OUTPUT_DIR)
print('Calibration objective:', calibration.objective)
print('Calibration gate passed:', calibration.passes_gate)
print('Simulated moments:', calibration.moments.as_dict())
print('Важно: targets — моделирующие допущения. При появлении реальных данных их следует заменить эмпирическими статистиками.')

In [ ]:
validation_seeds = list(range(config.evaluation.seed_start, config.evaluation.seed_start + config.evaluation.seeds))
stress_seeds = list(range(30_000, 30_000 + config.evaluation.stress_seeds))
scenario_seeds = {'base': validation_seeds, 'calm': validation_seeds, 'crisis_stress': stress_seeds}
result = evaluate_strategies(
    make_scenarios(config),
    make_default_strategies(config),
    seeds=scenario_seeds,
)
for row in result.summary:
    print(f"{row['scenario']:14s} {row['strategy']:18s} return={row['total_return_mean']:8.2%}  maxDD={row['max_drawdown_mean']:8.2%}  Sharpe={row['sharpe_mean']:7.3f}  reward={row['total_reward_mean']:8.3f}")
print('Reward diagnostics:', result.reward_diagnostics)
grid_count = config.evaluation.reward_grid_seeds
grid_seeds = {
    'base': validation_seeds[:grid_count],
    'calm': validation_seeds[:grid_count],
    'crisis_stress': stress_seeds[:grid_count],
}
grid_result = evaluate_strategies(
    make_scenarios(config),
    make_ppo_feasible_grid_strategies(config),
    seeds=grid_seeds,
)
reward_gate, reward_grid_ranking = build_reward_gate(result, grid_result, config)
print('Reward gate passed:', reward_gate['passes_reward_gate'])
print('Reward gate checks:', reward_gate['checks'])
print('Best feasible static policy:', reward_gate['best_grid_policy'])

In [ ]:
save_evaluation(result, OUTPUT_DIR)
save_evaluation(grid_result, OUTPUT_DIR / 'reward_grid')
save_reward_gate(reward_gate, reward_grid_ranking, OUTPUT_DIR)
display(Image(filename=str(OUTPUT_DIR / 'baseline_distributions.png')))
print('Artifacts saved to:', OUTPUT_DIR)
print(sorted(path.name for path in OUTPUT_DIR.iterdir()))

## Gate перед PPO

Переходим к PPO только если: полный запуск завершён без ошибок; `reward_gate.json` имеет `passes_reward_gate=true`; PPO-feasible reward/utility correlation проходит порог; robust optimum не имеет margin calls/bankruptcies; train/validation/test seed-диапазоны остаются непересекающимися. Stock boundary, использование futures и преимущество статического hedge сохраняются как diagnostics: они описывают reward landscape, но не заставляют статическую фьючерсную позицию выигрывать вопреки данным. Пользу динамического hedge должен доказать PPO validation gate. Результаты test-выборки не используются при настройке PPO.